In [1]:
from pgl import pgl
from pgl import pglFlywheel
from pgl import pglDatabase
from pgl import pglActions

import os

pgl = pgl()
pgl.cleanUp()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl->pglBase:checkOS) Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
(pgl->pglBase:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.6.2
(pgl->pglBase:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl->pglBase:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl->pglBase:checkOS)    Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pgl->pglBase:__init__) Main library instance created
(pgl->pglBase:removeOrphaned

---

Login to flywheel

---

In [2]:
# login to flywheel with apikey
apikey = os.environ['FW_API_KEY']
flywheel = pglFlywheel(apikey=apikey)

(pglFlywheel:__init__) Logged in as: Justin Gardner (jlg@stanford.edu)


---

Choose files to download from flywheel

---

In [3]:
flywheel.choose()

(pglFlywheelProject:loadedSessions) Loading sessions: mglretinotopy
(pglFlywheelSession:loadedScans) Loading scans: 2018-04-16 20:03
(pglFlywheelProject:loadedSessions) Loading sessions: Multimodal_Imaging
(pglFlywheelSession:loadedScans) Loading scans: 2025-08-28 23:11
(pglFlywheelSession:loadedScans) Loading scans: 2026-10-31 00:14
(pglFlywheelSession:loadedScans) Loading scans: 2026-10-30 20:39


---

Display selections

---

In [ ]:
downloaded = flywheel.download(dataPath="~/data/deleteme/flywheel-test-005")

In [ ]:
pglDatabase.print()

In [ ]:
behaviorSession = pglDatabase.loadSession(97)

In [ ]:
cachePath = Path("~/data/deleteme").expanduser() / f"flywheel-action-test-{uuid4().hex}"
loadFlywheelSession = pglActions.loadFlywheelSession()
loadFlywheelSession.configure(flywheel=flywheel, cacheDirectory=cachePath, experimentName="Flywheel action integration test", choose=True)

In [ ]:
session = loadFlywheelSession.run()

In [ ]:
session.print()

In [ ]:
session.saveRawSession()

In [ ]:
pglDatabase.print()

In [ ]:
session = pglDatabase.loadSession(120)

In [ ]:
session.mri.scans[0].data

In [ ]:
from pathlib import Path
from uuid import uuid4

from pgl.pglActions import pglActions
from pgl.pglPipeline import pglActionStatus

cachePath = Path("~/data/deleteme").expanduser() / f"flywheel-action-test-{uuid4().hex}"

action = pglActions.loadFlywheelSession()
action.configure(
    flywheel=flywheel,
    cacheDirectory=cachePath,
    experimentName="Flywheel action integration test",
    choose=True,
)
assert action.status == pglActionStatus.CONFIGURED

session = action.run()
assert session is not None
assert action.status == pglActionStatus.SUCCESS
assert not session.runs
assert session.mne is None
assert len(session.mri.scans) == len(action.selectedScans)
assert not cachePath.exists()

for scan in session.mri.scans:
    assert scan._data is None
    assert scan.localPath == ""

entry = session.actionHistory[-1]
assert entry.actionStatus == pglActionStatus.SUCCESS
assert set(entry.actionSettings) == {"selectedScans", "experimentName", "subjectID"}
assert entry.actionSettings["selectedScans"] == action.selectedScans
assert entry.actionSettings["selectedScans"] is not action.selectedScans

# Check that history encodes without trying to serialize a client/backing.
historyJSON = entry.toJSON(strict=True)
assert isinstance(historyJSON, str)

session.print()
print("Deferred Flywheel action and history checks passed.")

In [ ]:
!python -m py_compile /Users/justin/proj/pgl/pgl/pglDatabase.py

In [ ]:
print(inspect.getsource(flywheel.fw.acquisitions_api.download_file_from_acquisition))

In [ ]:
from pathlib import Path

assert len(downloaded) == 1
record = downloaded[0]
assert Path(record["localPath"]).stat().st_size == record["size_bytes"]
assert Path("~/data/deleteme/flywheel-test-001/flywheel.json").expanduser().is_file()
print(record["localPath"])
print(record["size_bytes"], record["sha256"])

In [ ]:
print(choose)

In [ ]:
projects = fw.projects()
for proj in projects:
    print(proj.label, proj.id)

# pick one project (equivalent to studyName in the matlab code)
project = fw.projects.find_first('label="mglretinotopy"')

sessions = project.sessions()
for sess in sessions:
    print(sess.label, sess.subject.label, sess.id)

In [ ]:

session = fw.get_session(sess.id)  # or just use sess directly
acquisitions = session.acquisitions()

for acq in acquisitions:
    print(acq.label, acq.id)
    for f in acq.files:
        print('   file:', f.name, f.type)

In [ ]:
acq = fw.get_acquisition(acq.id)  # refresh to be sure files are populated
file_obj = acq.files[0]  # or find_first by name/type

acq.download_file(file_obj.name, '/local/path/' + file_obj.name)

In [ ]:
import inspect
import pylink
print(inspect.getsource(pylink.EyeLinkCustomDisplay.draw_cross_hair))
print(inspect.getsource(pylink.EyeLinkCustomDisplay.draw_line))
print(inspect.getmembers(
    pylink.EyeLinkCustomDisplay,
    predicate=inspect.isfunction
))

import pylink

print("PyLink location:", pylink.__file__)
print("PyLink version:", getattr(pylink, "__version__", "unknown"))

import pylink.__version__ as v

print(v.__file__)
print(dir(v))
print(v.__version__)
print(inspect.getsource(pglEyeTrackingCalibrationTask))